# Phase 3: Multimodal Fusion and Ablation

Phase 3 concerns fusion architectures and per-modality ablations. It supports cached deep embeddings from Phase 2, and it also includes a lightweight real-data baseline that can run before BERT/ResNet embeddings are cached.

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from src.data import get_paths, load_train_data, make_stratified_split
from src.fusion_data import FusionFeatureSet, build_cached_embedding_fusion_features, build_lightweight_fusion_features
from src.fusion_model import build_fusion_model
from src.train import MultimodalTensorDataset, MultimodalTrainer
from src.utils import classification_metrics, ensure_dir, seed_everything

seed_everything(42)
paths = get_paths(root=PROJECT_ROOT)
train_df = load_train_data(paths)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Train shape: {train_df.shape}")
print(f"Device: {device}")

Train shape: (14993, 24)
Device: cuda


## Run Configuration

Defaults are intentionally bounded. Increase `FUSION_SAMPLE_SIZE`, `FUSION_EPOCHS`, and `ARCHITECTURES` after the lightweight run is stable.

In [9]:
RUN_CACHED_DEEP_FUSION = True
RUN_LIGHTWEIGHT_FUSION = False
RUN_ABLATIONS = True

FUSION_SAMPLE_SIZE = None
TEXT_COMPONENTS = 64
FUSION_BATCH_SIZE = 128
FUSION_EPOCHS = 12
FUSION_PATIENCE = 4
FUSION_LEARNING_RATE = 1e-3
ARCHITECTURES = ["early"]

feature_dir = ensure_dir(paths.outputs_dir / "features")
report_dir = ensure_dir(paths.outputs_dir / "reports")
checkpoint_dir = ensure_dir(paths.outputs_dir / "checkpoints")
bert_embedding_path = feature_dir / "bert_description_train.npz"
resnet_embedding_path = feature_dir / "resnet50_train.npz"

## Cached Deep Embeddings Path

Once Phase 2 generates `outputs/features/*_train.npz` files with `pet_ids` and `embeddings`, use `src.fusion_data.load_npz_embeddings` to align them by `PetID`. Until then, the lightweight baseline below uses real tabular features, TF-IDF/SVD text embeddings, and image-availability features from the image folder.

## Build Aligned Fusion Features

In [10]:
if RUN_CACHED_DEEP_FUSION:
    fusion_df = train_df
    if FUSION_SAMPLE_SIZE is not None and FUSION_SAMPLE_SIZE < len(train_df):
        _, fusion_df = train_test_split(
            train_df,
            test_size=FUSION_SAMPLE_SIZE,
            stratify=train_df["AdoptionSpeed"],
            random_state=42,
        )
        fusion_df = fusion_df.reset_index(drop=True)

    fusion_train_df, fusion_val_df = make_stratified_split(
        fusion_df,
        test_size=0.2,
        random_state=42,
    )
    train_features, val_features, coverage = build_cached_embedding_fusion_features(
        fusion_train_df,
        fusion_val_df,
        text_embedding_path=bert_embedding_path,
        image_embedding_path=resnet_embedding_path,
    )
    print("Using cached BERT + ResNet50 embeddings")
    print(coverage)
elif RUN_LIGHTWEIGHT_FUSION:
    fusion_df = train_df
    if FUSION_SAMPLE_SIZE is not None and FUSION_SAMPLE_SIZE < len(train_df):
        _, fusion_df = train_test_split(
            train_df,
            test_size=FUSION_SAMPLE_SIZE,
            stratify=train_df["AdoptionSpeed"],
            random_state=42,
        )
        fusion_df = fusion_df.reset_index(drop=True)

    fusion_train_df, fusion_val_df = make_stratified_split(
        fusion_df,
        test_size=0.2,
        random_state=42,
    )
    train_features, val_features = build_lightweight_fusion_features(
        fusion_train_df,
        fusion_val_df,
        paths,
        text_components=TEXT_COMPONENTS,
    )
else:
    raise ValueError("Enable RUN_CACHED_DEEP_FUSION or RUN_LIGHTWEIGHT_FUSION")

print(f"Train rows: {len(train_features.labels)}")
print(f"Val rows: {len(val_features.labels)}")
print(f"Tabular dim: {train_features.tabular_dim}")
print(f"Image dim: {train_features.image_dim}")
print(f"Text dim: {train_features.text_dim}")

Using cached BERT + ResNet50 embeddings
{'train_text_present': 11994, 'val_text_present': 2999, 'train_image_present': 11994, 'val_image_present': 2999, 'train_rows': 11994, 'val_rows': 2999}
Train rows: 11994
Val rows: 2999
Tabular dim: 28
Image dim: 2048
Text dim: 768


## Fusion Training Helpers

In [11]:
def make_fusion_loader(features: FusionFeatureSet, batch_size: int, shuffle: bool) -> DataLoader:
    dataset = MultimodalTensorDataset(
        features.tabular,
        features.image,
        features.text,
        features.labels,
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)


def run_fusion_experiment(
    architecture: str,
    train_set: FusionFeatureSet,
    val_set: FusionFeatureSet,
    run_name: str,
) -> tuple[dict, pd.DataFrame]:
    train_loader = make_fusion_loader(train_set, FUSION_BATCH_SIZE, shuffle=True)
    val_loader = make_fusion_loader(val_set, FUSION_BATCH_SIZE, shuffle=False)

    model = build_fusion_model(
        architecture,
        train_set.tabular_dim,
        train_set.image_dim,
        train_set.text_dim,
        num_classes=5,
    )
    trainer = MultimodalTrainer(
        model,
        device=device,
        labels=train_set.labels,
        learning_rate=FUSION_LEARNING_RATE,
        checkpoint_dir=checkpoint_dir,
        checkpoint_name=f"{run_name}_best.pt",
    )
    history = trainer.fit(
        train_loader,
        val_loader,
        epochs=FUSION_EPOCHS,
        patience=FUSION_PATIENCE,
    )
    val_loss, val_qwk, y_pred, y_true = trainer.validate(val_loader)
    metrics = classification_metrics(y_true, y_pred)
    summary = {
        "run_name": run_name,
        "architecture": architecture,
        "best_qwk": trainer.best_qwk,
        "final_qwk": val_qwk,
        "final_accuracy": metrics["accuracy"],
        "final_val_loss": val_loss,
        "epochs_ran": len(history),
    }
    return summary, pd.DataFrame(history)

## Fusion Architecture Runs

In [12]:
if "train_features" not in globals() or "val_features" not in globals():
    raise RuntimeError("Run the 'Build Aligned Fusion Features' cell before training fusion models.")

fusion_summaries = []
histories = {}

if RUN_CACHED_DEEP_FUSION or RUN_LIGHTWEIGHT_FUSION:
    for architecture in ARCHITECTURES:
        run_name = f"fusion_{architecture}_deep" if RUN_CACHED_DEEP_FUSION else f"fusion_{architecture}_lightweight"
        summary, history = run_fusion_experiment(
            architecture,
            train_features,
            val_features,
            run_name,
        )
        fusion_summaries.append(summary)
        histories[run_name] = history

fusion_results = pd.DataFrame(fusion_summaries)
display(fusion_results)
if not fusion_results.empty:
    output_name = "fusion_architecture_deep.csv" if RUN_CACHED_DEEP_FUSION else "fusion_architecture_lightweight.csv"
    fusion_results.to_csv(report_dir / output_name, index=False)

,run_name,architecture,best_qwk,final_qwk,final_accuracy,final_val_loss,epochs_ran
0,fusion_early_deep,early,0.305145,0.301264,0.37079,2.073644,7


## Ablation Study

Each ablation trains a fresh model with one modality zeroed in both train and validation features. This measures whether the model can benefit from that modality under the same split and training recipe.

In [13]:
if "train_features" not in globals() or "val_features" not in globals():
    raise RuntimeError("Run the 'Build Aligned Fusion Features' cell before running ablations.")

ablation_summaries = []

if (RUN_CACHED_DEEP_FUSION or RUN_LIGHTWEIGHT_FUSION) and RUN_ABLATIONS:
    architecture = ARCHITECTURES[0]
    ablations = {
        "full": (),
        "no_tabular": ("tabular",),
        "no_image": ("image",),
        "no_text": ("text",),
    }
    for ablation_name, blocked_modalities in ablations.items():
        ablated_train = train_features.without(*blocked_modalities)
        ablated_val = val_features.without(*blocked_modalities)
        prefix = "deep" if RUN_CACHED_DEEP_FUSION else "lightweight"
        run_name = f"fusion_{architecture}_{prefix}_{ablation_name}"
        summary, history = run_fusion_experiment(
            architecture,
            ablated_train,
            ablated_val,
            run_name,
        )
        summary["ablation"] = ablation_name
        summary["blocked_modalities"] = ",".join(blocked_modalities)
        ablation_summaries.append(summary)
        histories[run_name] = history

ablation_results = pd.DataFrame(ablation_summaries)
if not ablation_results.empty:
    full_best = ablation_results.loc[ablation_results["ablation"] == "full", "best_qwk"].iloc[0]
    ablation_results["delta_vs_full"] = ablation_results["best_qwk"] - full_best
    display(ablation_results.sort_values("best_qwk", ascending=False))
    output_name = "fusion_deep_ablation.csv" if RUN_CACHED_DEEP_FUSION else "fusion_lightweight_ablation.csv"
    ablation_results.to_csv(report_dir / output_name, index=False)

,run_name,architecture,best_qwk,final_qwk,final_accuracy,final_val_loss,epochs_ran,ablation,blocked_modalities,delta_vs_full
0,fusion_early_deep_full,early,0.328506,0.269352,0.377126,2.902162,10,full,,0.000000
1,fusion_early_deep_no_tabular,early,0.287852,0.268806,0.342114,2.290377,7,no_tabular,tabular,-0.040654
3,fusion_early_deep_no_text,early,0.286913,0.227301,0.311104,2.403057,11,no_text,text,-0.041593
2,fusion_early_deep_no_image,early,0.266686,0.244221,0.324441,1.800918,12,no_image,image,-0.061820
